# Chapter 4 — Transformer-Based Sentiment, Emotion and Stance

## Why this chapter matters

These methods turn social-media traces into careful, measurable evidence without treating posts as the whole population.

## Learning outcomes

- Explain tokenization, attention, pretrained and fine-tuned inference.
- Interpret sentiment, emotion, and zero-shot stance scores cautiously.

## Concepts in simple language

Transformers turn tokens into IDs and use attention to combine context. Pretraining learns general patterns; fine-tuning adapts a task. Maximum length and truncation constrain input. Domain shift and bias persist. A high score is model confidence, not research certainty.

## Dataset introduction

A tiny subset limits CPU work. Public models require no token. If downloads fail, the notebook uses a documented deterministic teaching fallback. All data are synthetic teaching data; they contain no real private users.

In [1]:
from pathlib import Path
import random, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
RANDOM_SEED = 42
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)
current_path = Path.cwd()
if (current_path / 'data').exists(): repository_root = current_path
elif (current_path.parent / 'data').exists(): repository_root = current_path.parent
else: raise FileNotFoundError('Could not find the repository data directory.')
data_directory = repository_root / 'data'
print('Python:', sys.version.split()[0], '| pandas:', pd.__version__)
print('Data directory:', data_directory)


Python: 3.9.6 | pandas: 2.2.3
Data directory: /Users/mehrdadjalali/Documents/SRH-Courses/Modules/Social Media Analytics and Sentiment Analysis/Github/SocialMediaAnayltics/data


## Instructor-guided walkthrough

`pipeline` uses `task`, `model`, `tokenizer`, and `device=-1` (CPU). Calls use `truncation=True`, `max_length=128`, and `top_k=None` to return all labels. Zero-shot candidate labels define the stance choices and strongly affect results.

## Step-by-step Python implementation

### Function guide: `transformers.pipeline`

**Library:** Hugging Face Transformers  
**Purpose:** Create an inference pipeline.  
**Input/return:** Takes the objects shown below and returns the stated result.  
**Common mistake:** Interpreting a score as causal proof.  
**Simple example:** See the next small code cell.

| Parameter | Value used | Meaning | What happens if changed? |
|---|---:|---|---|
| task | text-classification | Output type | zero-shot-classification changes the head |
| model/tokenizer | public model ID | Weights and token rules | Another model changes labels and behavior |
| device | -1 | CPU | 0 requests a GPU |
| truncation | True | Cut long inputs | False may exceed model limits |
| max_length | 128 | Token cap | Higher is slower |
| top_k | None | Return all label scores | 1 returns only the best |

In [2]:
import os
os.environ.setdefault('HF_HUB_OFFLINE', '1')  # remove or set to 0 when model downloads are allowed
sample=pd.read_csv(data_directory/'social_media_sample.csv').dropna(subset=['text']).head(5)
texts=sample['text'].tolist()
print('First run may download public model files. No authentication token is required.')
try:
    from transformers import pipeline
    sentiment_pipe=pipeline('text-classification',model='hf-internal-testing/tiny-random-distilbert',tokenizer='hf-internal-testing/tiny-random-distilbert',device=-1)
    raw=sentiment_pipe(texts,truncation=True,max_length=128,top_k=None)
    mode='transformer'
except Exception as exc:
    print('Model download unavailable; using deterministic teaching fallback:',type(exc).__name__)
    raw=[[{'label':'POSITIVE','score':.75 if 'great' in t.lower() or 'love' in t.lower() else .25},{'label':'NEGATIVE','score':.25 if 'great' in t.lower() or 'love' in t.lower() else .75}] for t in texts]
    mode='fallback'
print('Mode:',mode,'Raw first output:',raw[0])
sorted_outputs=[sorted(item,key=lambda x:x['score'],reverse=True) for item in raw]
display(pd.DataFrame({'text':texts,'dataset_label':sample['sentiment'],'model_top_label':[x[0]['label'] for x in sorted_outputs],'model_score':[round(x[0]['score'],3) for x in sorted_outputs]}))

First run may download public model files. No authentication token is required.


/Users/mehrdadjalali/Library/Python/3.9/lib/python/site-packages/google/api_core/_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.9.6). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/Users/mehrdadjalali/Library/Python/3.9/lib/python/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/mehrdadjalali/Library/Python/3.9/lib/python/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google 

Model download unavailable; using deterministic teaching fallback: AttributeError
Mode: fallback Raw first output: [{'label': 'POSITIVE', 'score': 0.25}, {'label': 'NEGATIVE', 'score': 0.75}]


,text,dataset_label,model_top_label,model_score
0,"Fast, friendly, and reliable transport support!",positive,NEGATIVE,0.75
1,The electric vehicles meeting starts at 10:00....,neutral,NEGATIVE,0.75
2,"Great, another broken university app... just w...",negative,POSITIVE,0.75
3,"Fast, friendly, and reliable customer service ...",positive,NEGATIVE,0.75
4,The public health meeting starts at 10:00. Mor...,neutral,NEGATIVE,0.75


The raw output is a list per text, containing label/score dictionaries. Tiny test weights demonstrate mechanics, not accuracy. For emotion, a task-fine-tuned model would return labels such as joy or anger. For stance, zero-shot candidate labels such as `support`, `oppose`, and `neutral` define the hypotheses.

In [3]:
candidate_labels=['support','oppose','neutral']
def teaching_zero_shot(text: str, labels: list[str]) -> list[dict]:
    scores={label:.2 for label in labels}; low=text.lower()
    scores['support']+=.5 if any(x in low for x in ['love','great','excellent']) else 0
    scores['oppose']+=.5 if any(x in low for x in ['broken','fail','disappointed']) else 0
    total=sum(scores.values()); return sorted([{'label':k,'score':v/total} for k,v in scores.items()],key=lambda x:x['score'],reverse=True)
print('Zero-shot stance structure:',teaching_zero_shot('I support this excellent plan',candidate_labels))
print('Ambiguous sarcasm:',teaching_zero_shot('Great, another broken app...',candidate_labels))

Zero-shot stance structure: [{'label': 'support', 'score': 0.6363636363636364}, {'label': 'oppose', 'score': 0.18181818181818185}, {'label': 'neutral', 'score': 0.18181818181818185}]
Ambiguous sarcasm: [{'label': 'support', 'score': 0.4375}, {'label': 'oppose', 'score': 0.4375}, {'label': 'neutral', 'score': 0.12500000000000003}]


## Output interpretation

Read values in context: inspect examples, sample sizes, uncertainty, and possible bias. Association is not causation.

## Common mistakes

Common errors include ignoring missing data, leakage, class imbalance, direction of edges, and treating model confidence as research certainty.

## Challenge 1: Compare transformer and VADER predictions

**Objective:** Apply a method demonstrated above.

**Tasks:** Adapt the starter code, check the result, and write two evidence-based sentences.

**Starter code:** `# result = ...`

**Hint:** Reuse the named columns and functions from the walkthrough.

**Expected output:** A small table or metric with no missing explanation.

**Interpretation question:** What does the result support, and what can it not prove?

**Self-check:** Did you report the sample size, units, and one limitation?

## Challenge 2: Test five difficult social media posts

**Objective:** Apply a method demonstrated above.

**Tasks:** Adapt the starter code, check the result, and write two evidence-based sentences.

**Starter code:** `# result = ...`

**Hint:** Reuse the named columns and functions from the walkthrough.

**Expected output:** A small table or metric with no missing explanation.

**Interpretation question:** What does the result support, and what can it not prove?

**Self-check:** Did you report the sample size, units, and one limitation?

## Challenge 3: Revise zero-shot stance candidate labels

**Objective:** Apply a method demonstrated above.

**Tasks:** Adapt the starter code, check the result, and write two evidence-based sentences.

**Starter code:** `# result = ...`

**Hint:** Reuse the named columns and functions from the walkthrough.

**Expected output:** A small table or metric with no missing explanation.

**Interpretation question:** What does the result support, and what can it not prove?

**Self-check:** Did you report the sample size, units, and one limitation?

## Chapter summary

The walkthrough connected a research question to transparent data preparation, analysis, visual evidence, and cautious interpretation.

## Connection to the group project

Validate models against human-coded data and document model, version, domain, and fallback.